In [1]:
# ================================================================
# PHASE 05.03 — COMMON-DIMENSIONAL PROJECTION
# ================================================================

print("=" * 70)
print("PHASE 05.03 — COMMON-DIMENSIONAL PROJECTION")
print("=" * 70)

print()
print("Objective:")
print("Convert the frozen 128-D modality representations")
print("into a shared 32-D representation space.")
print()
print("Pipeline:")
print("Genetic       : 128-D -> 64-D -> 32-D")
print("Behavioral    : 128-D -> 64-D -> 32-D")
print("Environmental : 128-D -> 64-D -> 32-D")
print()
print("Training performed : NO")
print("Checkpoint modified: NO")
print("Fusion performed    : NO")
print("Cross-modal inference: NO")
print("=" * 70)

PHASE 05.03 — COMMON-DIMENSIONAL PROJECTION

Objective:
Convert the frozen 128-D modality representations
into a shared 32-D representation space.

Pipeline:
Genetic       : 128-D -> 64-D -> 32-D
Behavioral    : 128-D -> 64-D -> 32-D
Environmental : 128-D -> 64-D -> 32-D

Training performed : NO
Checkpoint modified: NO
Fusion performed    : NO
Cross-modal inference: NO


In [2]:
# ================================================================
# CELL 2 — IMPORT LIBRARIES
# ================================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

print("Libraries imported successfully.")
print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

Libraries imported successfully.
PyTorch version: 2.14.0+cpu
NumPy version: 2.2.6
Pandas version: 2.2.3


In [3]:
# ================================================================
# CELL 3 — PROJECT PATHS
# ================================================================

PROJECT_ROOT = Path(
    r"C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening"
)

PHASE05_ROOT = (
    PROJECT_ROOT
    / "06_cross_modal_learning"
    / "05_multimodal_integration"
)

REPORT_DIR = PHASE05_ROOT / "reports" / "05_03_common_projection"

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:")
print(PROJECT_ROOT)

print()
print("Phase 05 root:")
print(PHASE05_ROOT)

print()
print("Report directory:")
print(REPORT_DIR)

Project root:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

Phase 05 root:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration

Report directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_03_common_projection


In [4]:
# ================================================================
# CELL 4 — ALIGNMENT CHECKPOINT PATHS
# ================================================================

GENETIC_ALIGNMENT_PATH = (
    PROJECT_ROOT
    / "Genetic_Model"
    / "04_cross_modal_alignment"
    / "genetic_128_to_32_alignment.pt"
)

BEHAVIORAL_ALIGNMENT_PATH = (
    PROJECT_ROOT
    / "Behavioral_Model"
    / "04_cross_modal_alignment"
    / "behavioral_128_to_32_alignment.pt"
)

ENVIRONMENTAL_ALIGNMENT_PATH = (
    PROJECT_ROOT
    / "Environmental_Model"
    / "04_cross_modal_alignment"
    / "environmental_128_to_32_alignment.pt"
)

alignment_paths = {
    "Genetic": GENETIC_ALIGNMENT_PATH,
    "Behavioral": BEHAVIORAL_ALIGNMENT_PATH,
    "Environmental": ENVIRONMENTAL_ALIGNMENT_PATH,
}

print("=" * 70)
print("ALIGNMENT CHECKPOINTS")
print("=" * 70)

for modality, path in alignment_paths.items():
    print()
    print(modality)
    print("Path:", path)
    print("Exists:", path.exists())

    assert path.exists(), f"Missing alignment checkpoint: {path}"

print()
print("[PASSED] All three alignment checkpoints exist.")

ALIGNMENT CHECKPOINTS

Genetic
Path: C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\Genetic_Model\04_cross_modal_alignment\genetic_128_to_32_alignment.pt
Exists: True

Behavioral
Path: C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\Behavioral_Model\04_cross_modal_alignment\behavioral_128_to_32_alignment.pt
Exists: True

Environmental
Path: C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\Environmental_Model\04_cross_modal_alignment\environmental_128_to_32_alignment.pt
Exists: True

[PASSED] All three alignment checkpoints exist.


In [5]:
# ================================================================
# CELL 5 — 128-D INPUT ARTIFACTS FROM PHASE 05.02
# ================================================================

INPUT_DIR = (
    PHASE05_ROOT
    / "reports"
    / "05_02_modality_inference"
)

GENETIC_128_PATH = (
    INPUT_DIR
    / "05_02_genetic_128d_embeddings.csv"
)

BEHAVIORAL_128_PATH = (
    INPUT_DIR
    / "05_02_behavioral_128d_embeddings.csv"
)

ENVIRONMENTAL_128_PATH = (
    INPUT_DIR
    / "05_02_environmental_128d_embeddings.csv"
)

input_paths = {
    "Genetic": GENETIC_128_PATH,
    "Behavioral": BEHAVIORAL_128_PATH,
    "Environmental": ENVIRONMENTAL_128_PATH,
}

print("=" * 70)
print("PHASE 05.02 INPUT ARTIFACTS")
print("=" * 70)

for modality, path in input_paths.items():
    print()
    print(modality)
    print("Path:", path)
    print("Exists:", path.exists())

    assert path.exists(), f"Missing 05.02 output: {path}"

print()
print("[PASSED] All 05.02 128-D embedding files found.")

PHASE 05.02 INPUT ARTIFACTS

Genetic
Path: C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_02_modality_inference\05_02_genetic_128d_embeddings.csv
Exists: True

Behavioral
Path: C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_02_modality_inference\05_02_behavioral_128d_embeddings.csv
Exists: True

Environmental
Path: C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_02_modality_inference\05_02_environmental_128d_embeddings.csv
Exists: True

[PASSED] All 05.02 128-D embedding files found.


In [6]:
# ================================================================
# CELL 6 — LOAD 128-D EMBEDDINGS
# ================================================================

genetic_128_df = pd.read_csv(GENETIC_128_PATH)
behavioral_128_df = pd.read_csv(BEHAVIORAL_128_PATH)
environmental_128_df = pd.read_csv(ENVIRONMENTAL_128_PATH)

print("=" * 70)
print("LOADED 128-D EMBEDDINGS")
print("=" * 70)

print()
print("Genetic shape:", genetic_128_df.shape)
print("Behavioral shape:", behavioral_128_df.shape)
print("Environmental shape:", environmental_128_df.shape)

LOADED 128-D EMBEDDINGS

Genetic shape: (5, 129)
Behavioral shape: (5, 129)
Environmental shape: (5, 129)


In [7]:
# ================================================================
# CELL 7 — VERIFY 128-D INPUT STRUCTURE
# ================================================================

def verify_128d_dataframe(df, modality):
    assert "patient_id" in df.columns, (
        f"{modality}: patient_id column missing."
    )

    embedding_columns = [
        column
        for column in df.columns
        if column != "patient_id"
    ]

    assert len(embedding_columns) == 128, (
        f"{modality}: expected 128 embedding columns, "
        f"found {len(embedding_columns)}."
    )

    values = df[embedding_columns].to_numpy(dtype=np.float32)

    assert values.ndim == 2
    assert values.shape[1] == 128

    assert np.isfinite(values).all(), (
        f"{modality}: NaN or Inf detected."
    )

    assert df["patient_id"].notna().all(), (
        f"{modality}: missing patient IDs."
    )

    return embedding_columns


genetic_128_columns = verify_128d_dataframe(
    genetic_128_df,
    "Genetic"
)

behavioral_128_columns = verify_128d_dataframe(
    behavioral_128_df,
    "Behavioral"
)

environmental_128_columns = verify_128d_dataframe(
    environmental_128_df,
    "Environmental"
)

print("=" * 70)
print("128-D INPUT VALIDATION")
print("=" * 70)

print("[PASSED] Genetic: 128-D")
print("[PASSED] Behavioral: 128-D")
print("[PASSED] Environmental: 128-D")
print()
print("[PASSED] No NaN or Inf values detected.")

128-D INPUT VALIDATION
[PASSED] Genetic: 128-D
[PASSED] Behavioral: 128-D
[PASSED] Environmental: 128-D

[PASSED] No NaN or Inf values detected.


In [8]:
# ================================================================
# CELL 8 — GENETIC 128-D -> 32-D PROJECTION
# ================================================================

class GeneticAlignment(nn.Module):
    """
    Exact Genetic 128-D -> 32-D alignment architecture.

    Architecture:
        128 -> 64 -> 32
    """

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 32),
        )

    def forward(self, x):
        return self.network(x)


genetic_projection = GeneticAlignment()

print(genetic_projection)

GeneticAlignment(
  (network): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=32, bias=True)
  )
)


In [9]:
# ================================================================
# CELL 9 — LOAD GENETIC PROJECTION CHECKPOINT
# ================================================================

genetic_checkpoint = torch.load(
    GENETIC_ALIGNMENT_PATH,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint type:")
print(type(genetic_checkpoint))

print()
print("Checkpoint keys:")
for key in genetic_checkpoint.keys():
    print(" ", key)

genetic_projection.load_state_dict(
    genetic_checkpoint
)

genetic_projection.eval()

print()
print("[PASSED] Genetic alignment checkpoint loaded.")

Checkpoint type:
<class 'collections.OrderedDict'>

Checkpoint keys:
  network.0.weight
  network.0.bias
  network.1.weight
  network.1.bias
  network.1.running_mean
  network.1.running_var
  network.1.num_batches_tracked
  network.3.weight
  network.3.bias

[PASSED] Genetic alignment checkpoint loaded.


In [10]:
# ================================================================
# CELL 10 — BEHAVIORAL 128-D -> 32-D PROJECTION
# ================================================================

class BehavioralAlignment(nn.Module):
    """
    Exact Behavioral alignment architecture.

    Encoder:
        128 -> 64 -> 32

    Decoder:
        32 -> 64 -> 128

    For Phase 05.03 we use only the encoder.
    """

    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 32),
        )

        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 128),
        )

    def forward(self, x):
        return self.encoder(x)


behavioral_projection = BehavioralAlignment()

print(behavioral_projection)

BehavioralAlignment(
  (encoder): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=32, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=128, bias=True)
  )
)


In [11]:
# ================================================================
# CELL 11 — LOAD BEHAVIORAL PROJECTION CHECKPOINT
# ================================================================

behavioral_checkpoint = torch.load(
    BEHAVIORAL_ALIGNMENT_PATH,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint type:")
print(type(behavioral_checkpoint))

print()
print("Top-level keys:")
for key in behavioral_checkpoint.keys():
    print(" ", key)

assert "model_state_dict" in behavioral_checkpoint

behavioral_projection.load_state_dict(
    behavioral_checkpoint["model_state_dict"]
)

behavioral_projection.eval()

print()
print("[PASSED] Behavioral alignment checkpoint loaded.")

Checkpoint type:
<class 'dict'>

Top-level keys:
  model_state_dict
  input_dimension
  output_dimension
  seed
  architecture
  purpose

[PASSED] Behavioral alignment checkpoint loaded.


In [12]:
# ================================================================
# CELL 12 — ENVIRONMENTAL 128-D -> 32-D PROJECTION
# ================================================================

class EnvironmentalAlignment(nn.Module):
    """
    Exact Environmental alignment architecture.

    Encoder:
        128 -> 64 -> 32

    Decoder:
        32 -> 64 -> 128

    Phase 05.03 uses only the encoder.
    """

    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 32),
        )

        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 128),
        )

    def forward(self, x):
        return self.encoder(x)


environmental_projection = EnvironmentalAlignment()

print(environmental_projection)

EnvironmentalAlignment(
  (encoder): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=32, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=128, bias=True)
  )
)


In [13]:
# ================================================================
# CELL 13 — LOAD ENVIRONMENTAL PROJECTION CHECKPOINT
# ================================================================

environmental_checkpoint = torch.load(
    ENVIRONMENTAL_ALIGNMENT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:")
print(type(environmental_checkpoint))

print()
print("Top-level keys:")
for key in environmental_checkpoint.keys():
    print(" ", key)

assert "model_state_dict" in environmental_checkpoint

environmental_projection.load_state_dict(
    environmental_checkpoint["model_state_dict"]
)

environmental_projection.eval()

print()
print("[PASSED] Environmental alignment checkpoint loaded.")

Checkpoint type:
<class 'dict'>

Top-level keys:
  model_state_dict
  input_dimension
  output_dimension
  normalization_mean
  normalization_std
  seed
  reconstruction_mse

[PASSED] Environmental alignment checkpoint loaded.


In [14]:
# ================================================================
# CELL 14 — PROJECTION MODEL VALIDATION
# ================================================================

def count_parameters(model):
    return sum(
        parameter.numel()
        for parameter in model.parameters()
    )


models = {
    "Genetic": genetic_projection,
    "Behavioral": behavioral_projection,
    "Environmental": environmental_projection,
}

print("=" * 70)
print("PROJECTION MODEL VALIDATION")
print("=" * 70)

for modality, model in models.items():

    model.eval()

    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    print()
    print(modality)
    print("Total parameters:", count_parameters(model))
    print("Trainable parameters:", trainable_parameters)

    assert trainable_parameters > 0

print()
print("[PASSED] Projection architectures loaded.")

PROJECTION MODEL VALIDATION

Genetic
Total parameters: 10464
Trainable parameters: 10464

Behavioral
Total parameters: 21024
Trainable parameters: 21024

Environmental
Total parameters: 21024
Trainable parameters: 21024

[PASSED] Projection architectures loaded.


In [15]:
# ================================================================
# CELL 15 — FREEZE PROJECTION MODELS
# ================================================================

for modality, model in models.items():

    model.eval()

    for parameter in model.parameters():
        parameter.requires_grad = False


print("=" * 70)
print("FROZEN MODEL VERIFICATION")
print("=" * 70)

for modality, model in models.items():

    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    print(
        f"{modality}: trainable parameters = "
        f"{trainable_parameters}"
    )

    assert trainable_parameters == 0

print()
print("[PASSED] All projection models are frozen.")
print("[PASSED] No projection training will occur.")

FROZEN MODEL VERIFICATION
Genetic: trainable parameters = 0
Behavioral: trainable parameters = 0
Environmental: trainable parameters = 0

[PASSED] All projection models are frozen.
[PASSED] No projection training will occur.


In [16]:
# ================================================================
# CELL 16 — FORWARD-PASS SANITY TEST
# ================================================================

dummy_input = torch.zeros(
    2,
    128,
    dtype=torch.float32
)

with torch.inference_mode():

    genetic_dummy_output = genetic_projection(
        dummy_input
    )

    behavioral_dummy_output = behavioral_projection(
        dummy_input
    )

    environmental_dummy_output = environmental_projection(
        dummy_input
    )


print("=" * 70)
print("FORWARD-PASS SANITY TEST")
print("=" * 70)

print()
print("Genetic output shape:",
      tuple(genetic_dummy_output.shape))

print("Behavioral output shape:",
      tuple(behavioral_dummy_output.shape))

print("Environmental output shape:",
      tuple(environmental_dummy_output.shape))


assert genetic_dummy_output.shape == (2, 32)
assert behavioral_dummy_output.shape == (2, 32)
assert environmental_dummy_output.shape == (2, 32)

print()
print("[PASSED] All three projection models output 32-D vectors.")

FORWARD-PASS SANITY TEST

Genetic output shape: (2, 32)
Behavioral output shape: (2, 32)
Environmental output shape: (2, 32)

[PASSED] All three projection models output 32-D vectors.


In [17]:
# ================================================================
# CELL 17 — PROJECTION HELPER FUNCTION
# ================================================================

def project_128d_to_32d(
    dataframe,
    embedding_columns,
    projection_model,
    modality
):
    """
    Convert a 128-D embedding DataFrame into a 32-D
    projected representation using a frozen model.
    """

    values = dataframe[embedding_columns].to_numpy(
        dtype=np.float32
    )

    assert values.ndim == 2
    assert values.shape[1] == 128

    assert np.isfinite(values).all(), (
        f"{modality}: input contains NaN or Inf."
    )

    tensor_input = torch.from_numpy(values)

    projection_model.eval()

    with torch.inference_mode():

        projected_tensor = projection_model(
            tensor_input
        )

    projected_array = (
        projected_tensor
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    assert projected_array.ndim == 2
    assert projected_array.shape[0] == len(dataframe)
    assert projected_array.shape[1] == 32

    assert np.isfinite(projected_array).all(), (
        f"{modality}: projection produced NaN or Inf."
    )

    return projected_array

In [18]:
# ================================================================
# CELL 18 — GENETIC 128-D -> 32-D
# ================================================================

genetic_32_array = project_128d_to_32d(
    dataframe=genetic_128_df,
    embedding_columns=genetic_128_columns,
    projection_model=genetic_projection,
    modality="Genetic"
)

print("=" * 70)
print("GENETIC PROJECTION")
print("=" * 70)

print("Input shape :", genetic_128_df.shape)
print("Output shape:", genetic_32_array.shape)

assert genetic_32_array.shape[1] == 32

print()
print("[PASSED] Genetic 128-D -> 32-D")

GENETIC PROJECTION
Input shape : (5, 129)
Output shape: (5, 32)

[PASSED] Genetic 128-D -> 32-D


In [19]:
# ================================================================
# CELL 19 — BEHAVIORAL 128-D -> 32-D
# ================================================================

behavioral_32_array = project_128d_to_32d(
    dataframe=behavioral_128_df,
    embedding_columns=behavioral_128_columns,
    projection_model=behavioral_projection,
    modality="Behavioral"
)

print("=" * 70)
print("BEHAVIORAL PROJECTION")
print("=" * 70)

print("Input shape :", behavioral_128_df.shape)
print("Output shape:", behavioral_32_array.shape)

assert behavioral_32_array.shape[1] == 32

print()
print("[PASSED] Behavioral 128-D -> 32-D")

BEHAVIORAL PROJECTION
Input shape : (5, 129)
Output shape: (5, 32)

[PASSED] Behavioral 128-D -> 32-D


In [20]:
# ================================================================
# CELL 20 — ENVIRONMENTAL 128-D -> 32-D
# ================================================================

environmental_32_array = project_128d_to_32d(
    dataframe=environmental_128_df,
    embedding_columns=environmental_128_columns,
    projection_model=environmental_projection,
    modality="Environmental"
)

print("=" * 70)
print("ENVIRONMENTAL PROJECTION")
print("=" * 70)

print("Input shape :", environmental_128_df.shape)
print("Output shape:", environmental_32_array.shape)

assert environmental_32_array.shape[1] == 32

print()
print("[PASSED] Environmental 128-D -> 32-D")

ENVIRONMENTAL PROJECTION
Input shape : (5, 129)
Output shape: (5, 32)

[PASSED] Environmental 128-D -> 32-D


In [21]:
# ================================================================
# CELL 21 — CREATE 32-D REPRESENTATION DATAFRAMES
# ================================================================

def create_32d_dataframe(
    original_dataframe,
    projected_array,
    prefix
):
    assert projected_array.shape[1] == 32

    columns = [
        f"{prefix}_{index:03d}"
        for index in range(1, 33)
    ]

    result = pd.DataFrame(
        projected_array,
        columns=columns
    )

    result.insert(
        0,
        "patient_id",
        original_dataframe["patient_id"].values
    )

    return result


genetic_32_df = create_32d_dataframe(
    genetic_128_df,
    genetic_32_array,
    "Genetic_32D"
)

behavioral_32_df = create_32d_dataframe(
    behavioral_128_df,
    behavioral_32_array,
    "Behavioral_32D"
)

environmental_32_df = create_32d_dataframe(
    environmental_128_df,
    environmental_32_array,
    "Environmental_32D"
)

print("=" * 70)
print("32-D DATAFRAMES")
print("=" * 70)

print()
print("Genetic:", genetic_32_df.shape)
print("Behavioral:", behavioral_32_df.shape)
print("Environmental:", environmental_32_df.shape)

32-D DATAFRAMES

Genetic: (5, 33)
Behavioral: (5, 33)
Environmental: (5, 33)


In [22]:
# ================================================================
# CELL 22 — PATIENT ID VERIFICATION
# ================================================================

genetic_ids = genetic_32_df["patient_id"].tolist()
behavioral_ids = behavioral_32_df["patient_id"].tolist()
environmental_ids = environmental_32_df["patient_id"].tolist()

print("=" * 70)
print("PATIENT ID VERIFICATION")
print("=" * 70)

print()
print("Genetic IDs:")
print(genetic_ids)

print()
print("Behavioral IDs:")
print(behavioral_ids)

print()
print("Environmental IDs:")
print(environmental_ids)

assert genetic_ids == behavioral_ids
assert behavioral_ids == environmental_ids

print()
print("[PASSED] Patient IDs are aligned across the three demo modalities.")

PATIENT ID VERIFICATION

Genetic IDs:
['P001', 'P002', 'P003', 'P004', 'P005']

Behavioral IDs:
['P001', 'P002', 'P003', 'P004', 'P005']

Environmental IDs:
['P001', 'P002', 'P003', 'P004', 'P005']

[PASSED] Patient IDs are aligned across the three demo modalities.


In [23]:
# ================================================================
# CELL 23 — FINAL 32-D VALIDATION
# ================================================================

def validate_32d_dataframe(df, modality):

    assert "patient_id" in df.columns

    embedding_columns = [
        column
        for column in df.columns
        if column != "patient_id"
    ]

    assert len(embedding_columns) == 32, (
        f"{modality}: expected 32 dimensions, "
        f"found {len(embedding_columns)}."
    )

    values = df[embedding_columns].to_numpy(
        dtype=np.float32
    )

    assert values.shape[1] == 32
    assert np.isfinite(values).all()
    assert df["patient_id"].notna().all()

    return {
        "rows": int(values.shape[0]),
        "dimensions": int(values.shape[1]),
        "nan_count": int(np.isnan(values).sum()),
        "inf_count": int(np.isinf(values).sum()),
    }


genetic_validation = validate_32d_dataframe(
    genetic_32_df,
    "Genetic"
)

behavioral_validation = validate_32d_dataframe(
    behavioral_32_df,
    "Behavioral"
)

environmental_validation = validate_32d_dataframe(
    environmental_32_df,
    "Environmental"
)

print("=" * 70)
print("FINAL 32-D REPRESENTATION VALIDATION")
print("=" * 70)

print()
print("Genetic:")
print(genetic_validation)

print()
print("Behavioral:")
print(behavioral_validation)

print()
print("Environmental:")
print(environmental_validation)

print()
print("[PASSED] All three modalities contain valid 32-D representations.")

FINAL 32-D REPRESENTATION VALIDATION

Genetic:
{'rows': 5, 'dimensions': 32, 'nan_count': 0, 'inf_count': 0}

Behavioral:
{'rows': 5, 'dimensions': 32, 'nan_count': 0, 'inf_count': 0}

Environmental:
{'rows': 5, 'dimensions': 32, 'nan_count': 0, 'inf_count': 0}

[PASSED] All three modalities contain valid 32-D representations.


In [24]:
# ================================================================
# CELL 24 — SAVE 32-D REPRESENTATIONS
# ================================================================

GENETIC_32_OUTPUT = (
    REPORT_DIR
    / "05_03_genetic_32d_projection.csv"
)

BEHAVIORAL_32_OUTPUT = (
    REPORT_DIR
    / "05_03_behavioral_32d_projection.csv"
)

ENVIRONMENTAL_32_OUTPUT = (
    REPORT_DIR
    / "05_03_environmental_32d_projection.csv"
)

genetic_32_df.to_csv(
    GENETIC_32_OUTPUT,
    index=False
)

behavioral_32_df.to_csv(
    BEHAVIORAL_32_OUTPUT,
    index=False
)

environmental_32_df.to_csv(
    ENVIRONMENTAL_32_OUTPUT,
    index=False
)

print("=" * 70)
print("32-D REPRESENTATIONS SAVED")
print("=" * 70)

print()
print("Genetic:")
print(GENETIC_32_OUTPUT)

print()
print("Behavioral:")
print(BEHAVIORAL_32_OUTPUT)

print()
print("Environmental:")
print(ENVIRONMENTAL_32_OUTPUT)

32-D REPRESENTATIONS SAVED

Genetic:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_03_common_projection\05_03_genetic_32d_projection.csv

Behavioral:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_03_common_projection\05_03_behavioral_32d_projection.csv

Environmental:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_03_common_projection\05_03_environmental_32d_projection.csv


In [25]:
# ================================================================
# CELL 25 — CREATE PROJECTION REPORT
# ================================================================

projection_report = {
    "phase": "05.03",
    "notebook": "05_03_common_projection.ipynb",
    "objective": (
        "Project frozen 128-D modality representations "
        "into a common 32-D dimensional space."
    ),

    "training_performed": False,
    "checkpoint_modified": False,
    "fusion_performed": False,
    "cross_modal_inference_performed": False,

    "input_dimension": 128,
    "output_dimension": 32,

    "modalities": {
        "Genetic": {
            "architecture": "128 -> 64 -> 32",
            "checkpoint": str(GENETIC_ALIGNMENT_PATH),
            "input_rows": int(len(genetic_32_df)),
            "output_dimensions": 32,
        },

        "Behavioral": {
            "architecture": "128 -> 64 -> 32",
            "checkpoint": str(BEHAVIORAL_ALIGNMENT_PATH),
            "input_rows": int(len(behavioral_32_df)),
            "output_dimensions": 32,
        },

        "Environmental": {
            "architecture": "128 -> 64 -> 32",
            "checkpoint": str(ENVIRONMENTAL_ALIGNMENT_PATH),
            "input_rows": int(len(environmental_32_df)),
            "output_dimensions": 32,
        },
    },

    "outputs": {
        "Genetic": str(GENETIC_32_OUTPUT),
        "Behavioral": str(BEHAVIORAL_32_OUTPUT),
        "Environmental": str(ENVIRONMENTAL_32_OUTPUT),
    },
}

REPORT_PATH = (
    REPORT_DIR
    / "05_03_common_projection_report.json"
)

with open(
    REPORT_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        projection_report,
        file,
        indent=4
    )

print("=" * 70)
print("PROJECTION REPORT SAVED")
print("=" * 70)

print(REPORT_PATH)

PROJECTION REPORT SAVED
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_03_common_projection\05_03_common_projection_report.json


In [26]:
# ================================================================
# CELL 26 — FINAL PHASE 05.03 VALIDATION
# ================================================================

print("=" * 70)
print("PHASE 05.03 FINAL VALIDATION")
print("=" * 70)

output_files = {
    "Genetic": GENETIC_32_OUTPUT,
    "Behavioral": BEHAVIORAL_32_OUTPUT,
    "Environmental": ENVIRONMENTAL_32_OUTPUT,
    "Report": REPORT_PATH,
}

for name, path in output_files.items():

    assert path.exists(), (
        f"Missing output: {path}"
    )

    print(f"[PASSED] {name}: {path.name}")


print()

assert genetic_32_df.shape[1] == 33
assert behavioral_32_df.shape[1] == 33
assert environmental_32_df.shape[1] == 33

assert genetic_32_df.shape[0] == 5
assert behavioral_32_df.shape[0] == 5
assert environmental_32_df.shape[0] == 5

print("[PASSED] Genetic output: 5 × 32-D")
print("[PASSED] Behavioral output: 5 × 32-D")
print("[PASSED] Environmental output: 5 × 32-D")

print()

assert genetic_ids == behavioral_ids
assert behavioral_ids == environmental_ids

print("[PASSED] Patient IDs aligned across modalities.")

print()
print("=" * 70)
print("PHASE 05.03 STATUS: PASSED")
print("=" * 70)

print()
print("No training performed.")
print("No checkpoint modified.")

print()
print("Genetic:")
print("    128-D -> 64-D -> 32-D")

print()
print("Behavioral:")
print("    128-D -> 64-D -> 32-D")

print()
print("Environmental:")
print("    128-D -> 64-D -> 32-D")

print()
print("Next phase:")
print("    05.04 Cross-Modal Inference")

PHASE 05.03 FINAL VALIDATION
[PASSED] Genetic: 05_03_genetic_32d_projection.csv
[PASSED] Behavioral: 05_03_behavioral_32d_projection.csv
[PASSED] Environmental: 05_03_environmental_32d_projection.csv
[PASSED] Report: 05_03_common_projection_report.json

[PASSED] Genetic output: 5 × 32-D
[PASSED] Behavioral output: 5 × 32-D
[PASSED] Environmental output: 5 × 32-D

[PASSED] Patient IDs aligned across modalities.

PHASE 05.03 STATUS: PASSED

No training performed.
No checkpoint modified.

Genetic:
    128-D -> 64-D -> 32-D

Behavioral:
    128-D -> 64-D -> 32-D

Environmental:
    128-D -> 64-D -> 32-D

Next phase:
    05.04 Cross-Modal Inference
